In [1]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score
import pandas as pd

In [2]:
X_train = pd.read_csv('csv/predictors_withsom_scaled_train.csv')
X_test = pd.read_csv('csv/predictors_withsom_scaled_test.csv')
y_train = pd.read_csv('csv/target_train.csv')
y_test = pd.read_csv('csv/target_test.csv')

In [3]:
regions_train = y_train[['region']]
regions_test = y_test[['region']]
#X_train = X_train.drop('region', axis=1)
#X_test = X_test.drop('region', axis=1)
y_train = y_train.drop('region', axis=1)
y_test = y_test.drop('region', axis=1)

In [4]:
ridge_regressor = Ridge()
lasso_regressor = Lasso()

# Подбор параметра регуляризации с помощью кросс-валидации
> В датасете помимо исходных переменных присутствует закодированная (One-Hot) переменная кластерного деления объектов

## Сетка параметров

In [9]:
grid_params = {
    'alpha': [0.01, 0.05, 0.1, 0.5, 1, 5, 10]
}

## Ridge

### Перебор параметров по сетке

In [36]:
grid_ridge = GridSearchCV(ridge_regressor, grid_params, cv=5, verbose=1, scoring='r2')
grid_ridge.fit(X_train, y_train)

Fitting 5 folds for each of 7 candidates, totalling 35 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Ridge()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'alpha': [0.01, 0.05, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >3 : the fold and candidate parameter indexes are also 

### Лучшая модель

In [37]:
ridge_regressor = grid_ridge.best_estimator_
ridge_params = grid_ridge.best_params_

### Лучший параметр регуляризации

In [38]:
ridge_params['alpha']

10

### Предсказанные значения

In [39]:
y_pred = ridge_regressor.predict(X_test)
y_train_pred = ridge_regressor.predict(X_train)

In [40]:
response_test = pd.concat([regions_test, pd.Series(y_pred, name='ratio_underage_partic_predicted'), y_test], axis=1)
response_train = pd.concat([regions_train, pd.Series(y_train_pred, name='ratio_underage_partic_predicted'), y_train], axis=1)

In [50]:
response_train

,region,ratio_underage_partic_predicted,ratio_underage_partic
0,Камчатский край,0.590673,0.723200
1,Чеченская Республика,0.005275,0.003583
2,Чувашская Республика - Чувашия,0.289051,0.339983
3,Рязанская область,0.266490,0.253967
4,Республика Калмыкия,0.375474,0.266633
...,...,...,...
63,Ненецкий автономный округ,0.652151,0.709033
64,Ханты-Мансийский автономный округ - Югра,0.336773,0.215500
65,Новосибирская область,0.557687,0.619767
66,Тверская область,0.577046,0.495067


In [51]:
response_test

,region,ratio_underage_partic_predicted,ratio_underage_partic
0,Приморский край,0.638246,0.703667
1,Белгородская область,0.288785,0.243133
2,Красноярский край,0.699101,0.587017
3,Вологодская область,0.579057,0.591933
4,Смоленская область,0.574643,0.596300
5,Сахалинская область,0.761782,0.826383
6,Орловская область,0.426621,0.301567
7,Республика Карелия,0.631097,0.919950
8,Ивановская область,0.478875,0.500083
9,Республика Хакасия,0.581732,0.574600


### Метрики

In [41]:
ridge_metrics = pd.DataFrame({
    'MAE':[mean_absolute_error(y_pred, y_test), mean_absolute_error(y_train_pred, y_train)],
    'MAPE':[mean_absolute_percentage_error(y_pred, y_test), mean_absolute_percentage_error(y_train_pred, y_train)],
    'R2':[r2_score(y_pred, y_test), r2_score(y_train_pred, y_train)]
}, index = ['test', 'train'])

In [42]:
ridge_metrics

,MAE,MAPE,R2
test,0.080900,0.209338,0.553889
train,0.075364,0.172264,0.695623


### Коэффициенты

In [43]:
coeffs = pd.DataFrame(ridge_regressor.coef_, index=X_train.columns, columns=['coeffs'])
coeffs

,coeffs
health_ratio,-0.008872
coeff_Gini,-0.013334
not_smoking_male%,-0.062134
not_smoking_female%,-0.074541
num_of_criminals,-0.005527
num_of_alcohol_crimes,0.002287
num_of_female_crimes,-0.013848
num_of_drugs_crimes,0.005479
num_of_underaged_criminals,0.074744
life_quality,-0.079765


## Lasso

In [44]:
grid_lasso = GridSearchCV(lasso_regressor, grid_params, cv=5, scoring='r2', verbose=1)
grid_lasso.fit(X_train, y_train)

Fitting 5 folds for each of 7 candidates, totalling 35 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Lasso()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'alpha': [0.01, 0.05, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >3 : the fold and candidate parameter indexes are also 

### Лучшая модель

In [45]:
lasso_regressor = grid_lasso.best_estimator_
lasso_params = grid_lasso.best_params_

### Лучший параметр регуляризации

In [46]:
lasso_params['alpha']

0.01

### Предсказанные значения

In [47]:
y_pred = lasso_regressor.predict(X_test)
y_train_pred = lasso_regressor.predict(X_train)

In [48]:
response_test = pd.concat([regions_test, pd.Series(y_pred, name='ratio_underage_partic_predicted'), y_test], axis=1)
response_train = pd.concat([regions_train, pd.Series(y_train_pred, name='ratio_underage_partic_predicted'), y_train], axis=1)

### Метрики

In [49]:
lasso_metrics = pd.DataFrame({
    'MAE':[mean_absolute_error(y_pred, y_test), mean_absolute_error(y_train_pred, y_train)],
    'MAPE':[mean_absolute_percentage_error(y_pred, y_test), mean_absolute_percentage_error(y_train_pred, y_train)],
    'R2':[r2_score(y_pred, y_test), r2_score(y_train_pred, y_train)]
}, index = ['test', 'train'])
lasso_metrics

,MAE,MAPE,R2
test,0.085775,0.221241,0.501010
train,0.081627,0.187228,0.649349


# Подбор параметра регуляризации с помощью кросс-валидации
> В датасете помимо исходных переменных присутствует единая стандартизованная переменная кластерного деления объектов

In [5]:
X_train = pd.read_csv('csv/predictors_with_onesom_scaled_train.csv')
X_test = pd.read_csv('csv/predictors_with_onesom_scaled_test.csv')
y_train = pd.read_csv('csv/target_train.csv')
y_test = pd.read_csv('csv/target_test.csv')

In [6]:
regions_train = y_train[['region']]
regions_test = y_test[['region']]
#X_train = X_train.drop('region', axis=1)
#X_test = X_test.drop('region', axis=1)
y_train = y_train.drop('region', axis=1)
y_test = y_test.drop('region', axis=1)

In [7]:
ridge_regressor = Ridge()

In [10]:
grid_ridge = GridSearchCV(ridge_regressor, grid_params, cv=5, verbose=1, scoring='r2')
grid_ridge.fit(X_train, y_train)

Fitting 5 folds for each of 7 candidates, totalling 35 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Ridge()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'alpha': [0.01, 0.05, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >3 : the fold and candidate parameter indexes are also 

In [12]:
ridge_regressor = grid_ridge.best_estimator_
ridge_params = grid_ridge.best_params_

In [13]:
ridge_params['alpha']

5

In [14]:
y_pred = ridge_regressor.predict(X_test)
y_train_pred = ridge_regressor.predict(X_train)

In [15]:
response_test = pd.concat([regions_test, pd.Series(y_pred, name='ratio_underage_partic_predicted'), y_test], axis=1)
response_train = pd.concat([regions_train, pd.Series(y_train_pred, name='ratio_underage_partic_predicted'), y_train], axis=1)

In [16]:
ridge_metrics = pd.DataFrame({
    'MAE':[mean_absolute_error(y_pred, y_test), mean_absolute_error(y_train_pred, y_train)],
    'MAPE':[mean_absolute_percentage_error(y_pred, y_test), mean_absolute_percentage_error(y_train_pred, y_train)],
    'R2':[r2_score(y_pred, y_test), r2_score(y_train_pred, y_train)]
}, index = ['test', 'train'])

In [17]:
ridge_metrics

,MAE,MAPE,R2
test,0.073567,0.197236,0.612736
train,0.074917,0.188645,0.712961


In [64]:
coeffs = pd.DataFrame(ridge_regressor.coef_, index=X_train.columns, columns=['coeffs'])
coeffs

,coeffs
health_ratio,-0.000551
coeff_Gini,-0.010537
not_smoking_male%,-0.062791
not_smoking_female%,-0.083598
num_of_criminals,-0.009039
num_of_alcohol_crimes,-0.019151
num_of_female_crimes,-0.027239
num_of_drugs_crimes,0.016761
num_of_underaged_criminals,0.094350
life_quality,-0.091864


In [21]:
float(ridge_regressor.intercept_[0])

0.4770007352843136